In [ ]:
from utils import *
from plotly.subplots import make_subplots
from tqdm.auto import tqdm
import json

In [ ]:
def pretrain_loader():
    results_dir = Path("../results/pretrain")
    scalars = TBScalars(".cache/pretrain")

    res_df = []
    for test in tqdm([*results_dir.iterdir()]):
        params = test.name.split("-")
        test_r = {}
        test_r["env"] = params[0]
        types = {"wm_ratio": int, "rl_ratio": int, "rl_freq": float, "seed": int}
        for (name, typ), value in zip(types.items(), params[1:]):
            test_r[name] = typ(value.removeprefix(f"{name}="))
        df = scalars.read(test)
        scores = df[df["tag"] == "val/mean_ep_ret"]["value"]
        test_r["score"] = scores.iloc[-1]
        res_df.append({"path": test, **test_r})
    res_df = pd.DataFrame.from_records(res_df)

    tags = []
    for _, row in res_df.iterrows():
        tags.append(f"{row['wm_ratio']}/{row['rl_ratio']}")
    res_df["tag"] = tags

    return res_df, scalars


res_df, scalars = pretrain_loader()
res_df

In [ ]:
res_df0 = res_df[res_df["rl_freq"] == 0.0]
res_df1 = res_df[res_df["rl_freq"] != 0.0]

fig = go.Figure()
fig.add_trace(
    go.Violin(
        x=res_df0["tag"],
        y=res_df0["score"],
        side="negative",
        name="No RL opt",
        pointpos=-1.0,
        hovertext=res_df0["path"].apply(str),
    )
)
fig.add_trace(
    go.Violin(
        x=res_df1["tag"],
        y=res_df1["score"],
        side="positive",
        name="With RL opt",
        pointpos=1.0,
        hovertext=res_df1["path"].apply(str),
    )
)
fig.update_traces(meanline_visible=True, points="all")
fig

In [ ]:
def split_ratio_loader():
    results_dir = Path("../results/split_ratios")
    scalars = TBScalars(".cache/split_ratios")

    res_df = []
    for test in tqdm([*results_dir.iterdir()]):
        params = test.name.split("-")
        test_r = {}
        test_r["env"] = params[0]
        types = {"wm_ratio": int, "rl_ratio": int, "seed": int}
        for (name, typ), value in zip(types.items(), params[1:]):
            test_r[name] = typ(value.removeprefix(f"{name}="))
        df = scalars.read(test)
        df = df[df["tag"] == "val/mean_ep_ret"]
        test_r["score"] = df.iloc[-1]["value"]
        res_df.append({"path": test, **test_r})
        scalars.read(test)
    res_df = pd.DataFrame.from_records(res_df)

    tags = []
    for _, row in res_df.iterrows():
        tags.append(f"{row['wm_ratio']}/{row['rl_ratio']}")
    res_df["tag"] = tags

    return res_df, scalars


res_df_sr, _ = split_ratio_loader()
res_df_sr

In [ ]:
fig = go.Figure()
fig.add_trace(
    go.Violin(
        x=res_df0["tag"],
        y=res_df0["score"],
        side="negative",
        name="Pretraining",
        pointpos=-1.0,
    )
)
fig.add_trace(
    go.Violin(
        x=res_df_sr["tag"],
        y=res_df_sr["score"],
        side="positive",
        name="No Pretraining",
        pointpos=1.0,
    )
)
fig.update_traces(meanline_visible=True, points="all")
fig

In [ ]:
wm_ratios = res_df0["wm_ratio"].unique()
rl_ratios = res_df0["rl_ratio"].unique()

fig = make_subplots(
    rows=len(wm_ratios),
    row_titles=[str(x) for x in wm_ratios],
    cols=len(rl_ratios),
    column_titles=[str(x) for x in rl_ratios],
)

color = next(make_color_iter())
for row, wm_ratio in enumerate(wm_ratios, 1):
    for col, rl_ratio in enumerate(rl_ratios, 1):
        res_dfs = res_df0[
            (res_df0["wm_ratio"] == wm_ratio) & (res_df0["rl_ratio"] == rl_ratio)
        ]
        dfs = []
        for _, test in res_dfs.iterrows():
            df = scalars.read(test["path"])
            df = df[df["tag"] == "val/mean_ep_ret"]
            df["index"] = np.arange(len(df))
            dfs.append(df)
        df = pd.concat(dfs)

        g = df.groupby("index")
        avg_df = pd.DataFrame.from_records(
            {
                "score_mean": g["value"].mean(),
                "score_std": g["value"].std(),
                "step": g["step"].median(),
            }
        )

        for trace in err_line(
            x=avg_df["step"],
            y=avg_df["score_mean"],
            std=avg_df["score_std"],
            color=color,
        ):
            fig.add_trace(trace, row=row, col=col)

fig